In [7]:
%load_ext autoreload
%autoreload 2

Loop through each filter and all the models and determine the threshold values for each

In [8]:
from get_model_probabilities import *

import scienceplots
plt.style.use(["science","grid"])

Source redshift:1.36


# Base models 

In [11]:
imonte=1

#target_test_acc = pkl.load(open("pickles/target_test_accuracy.pkl","rb"))

for ifilter in ['concat']:
    
    all_models =  glob(f"../models/base_models/cdan_baha2dark_pre_squeezenet1_aw_1_pad_shear_avgpool_gauss_seed_*_nob1_final.pth")

    
    output_name = f"pickles/basemodel_sep_results.pkl"
    
    if os.path.isfile( output_name ):
        all_results = pkl.load(open(output_name,"rb"))
    else:
        all_results = {}
        
    for imodel in tqdm(all_models):

        seed = imodel.split('_')[-3]

        

        if f"seed_{seed}" in all_results.keys():
            continue
        args.jwst_filter = 'concat'
        args.apply_intrinsic_ell = 1.
        zs = {
        'f115w':1.36,
        'f150w':1.56,
        'concat':1.46
        }
            
        domain = {
        'tgt':'darkskies_obs',
        'src':'bahamas_obs'
        }
        args.ignore_dataset = [''] # Althopiugh i ignored during training i want to see duringn testing.
        all_results[f"seed_{seed}"]  = {'src':[],'tgt':[]}

        args.log_mass_cut=0. # Make sure i test on equal mass clusters
        args.apply_intrinsic_ell = 0
        args.unbalance = True
        for i in range(imonte):
            args.zs = zs['concat']

            for idomain in domain.keys():

                target_domain = domain[idomain]
                results = get_probabilities( 
                        target_domain,
                        [imodel],
                        args,
                        quiet=True
                )
                del results['data_loaders']

                all_results[f"seed_{seed}"][idomain].append( results )

        pkl.dump(all_results, open(output_name,"wb"))


100%|█████████████████████████████████████████| 30/30 [18:18<00:00, 36.61s/it]


# Final forward modelled results

In [27]:
imonte=40

#target_test_acc = pkl.load(open("pickles/target_test_accuracy.pkl","rb"))

for ifilter in ['concat','f115w','f150w']:
    
    all_models =  glob(f"../models/concat/cdan_baha2dark_pre_squeezenet1_aw_1_pad_shear_avgpool_gauss_seed_*_nob1_ft_tweaked_align_10_best.pth")

    
    output_name = f"pickles/all_models_{ifilter}_zs_results.pkl"
    

    all_results = pkl.load(open(output_name,'rb'))
        
    for imodel in tqdm(all_models):

        seed = imodel.split('_')[-7]

        


        args.jwst_filter = ifilter
        args.apply_intrinsic_ell = 1.

        zs = {
        'f115w':1.6,
        'f150w':1.65,
        'concat':1.65
        }
            
        domain = {
        'tgt':'darkskies_obs',
        'src':'bahamas_obs'
        }
        args.ignore_dataset = [''] # Although i ignored during training i want to see duringn testing.
                           
        if f"seed_{seed}" not in all_results.keys():
            
            all_results[f"seed_{seed}"]  = {'src':[],'tgt':[]}

        args.unbalance = True
                           
        for i in range(imonte):
            args.zs = zs[ifilter]

            for idomain in domain.keys():

                target_domain = domain[idomain]
                results = get_probabilities( 
                        target_domain,
                        [imodel],
                        args,
                        quiet=True
                )
                del results['data_loaders']

                all_results[f"seed_{seed}"][idomain].append( results )

        pkl.dump(all_results, open(output_name,"wb"))


100%|█████████████████████████████████████████| 30/30 [15:20:05<00:00, 1840.18s/it]
